# 6. Анализ ошибок (Error Analysis)

В этом ноутбуке проводим детальный анализ результатов финальной модели:

1. **Общие метрики** — MAE, RMSE, R², MAPE на тестовой выборке
2. **Распределение ошибок** — гистограмма остатков, Actual vs Predicted
3. **Где модель ошибается сильнее** — топ худших предсказаний
4. **Ошибки по категориям** — по производителю, возрасту, состоянию
5. **Важность признаков** — Permutation Importance
6. **Выводы и ограничения модели**

---

## 1. Загрузка данных и модели

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              r2_score, mean_absolute_percentage_error)
from sklearn.inspection import permutation_importance

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)

Path('../reports/figures').mkdir(parents=True, exist_ok=True)
print("Библиотеки загружены")

In [ ]:
data_path = Path('../data/processed')

X_test      = pd.read_csv(data_path / 'X_test.csv',   index_col=0)
y_test_log  = pd.read_csv(data_path / 'y_test.csv',   index_col=0).squeeze()
test_raw    = pd.read_csv(data_path / 'test_raw.csv',  index_col=0)

model = joblib.load('../models/final_model.pkl')

print(f"X_test shape:   {X_test.shape}")
print(f"test_raw shape: {test_raw.shape}")
print(f"\nМодель: {type(model).__name__}")
print(f"Колонки X_test: {X_test.columns.tolist()[:5]} ... (всего {X_test.shape[1]})")

In [ ]:
y_pred_log = model.predict(X_test)
y_pred     = np.expm1(y_pred_log)          # предсказания в долларах
y_true     = np.expm1(y_test_log.values)   # факт в долларах

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2   = r2_score(y_true, y_pred)
mape = mean_absolute_percentage_error(y_true, y_pred) * 100

print("=" * 45)
print("Метрики на тестовой выборке")
print("=" * 45)
print(f"  MAE:  ${mae:>10,.2f}")
print(f"  RMSE: ${rmse:>10,.2f}")
print(f"  R²:    {r2:>10.4f}")
print(f"  MAPE:  {mape:>10.2f}%")
print("=" * 45)

## 2. Распределение ошибок

### 2.1 Actual vs Predicted

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

ax.scatter(y_true, y_pred, alpha=0.3, s=15, color='steelblue', rasterized=True)

lim = max(y_true.max(), y_pred.max())
ax.plot([0, lim], [0, lim], 'r--', linewidth=2, label='Идеальное предсказание')

ax.set_xlabel('Фактическая цена ($)', fontsize=12)
ax.set_ylabel('Предсказанная цена ($)', fontsize=12)
ax.set_title('Actual vs Predicted', fontsize=14)
ax.legend(fontsize=11)

# Аннотация метрик прямо на графике
ax.text(0.05, 0.95, f'MAE = ${mae:,.0f}\nR² = {r2:.4f}',
        transform=ax.transAxes, fontsize=11,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('../reports/figures/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 Распределение остатков

In [ ]:
residuals = y_true - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Гистограмма остатков
axes[0].hist(residuals, bins=80, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(x=0,                    color='red',    linestyle='--', linewidth=2, label='0')
axes[0].axvline(x=residuals.mean(),     color='orange', linestyle='-',  linewidth=1.5, label=f'mean={residuals.mean():+,.0f}')
axes[0].axvline(x=np.median(residuals), color='green',  linestyle='-',  linewidth=1.5, label=f'median={np.median(residuals):+,.0f}')
axes[0].set_xlabel('Остаток (фактическая − предсказанная), $')
axes[0].set_ylabel('Частота')
axes[0].set_title('Распределение остатков', fontsize=13)
axes[0].legend()

# Остатки vs Predicted (поиск паттернов)
axes[1].scatter(y_pred, residuals, alpha=0.3, s=12, color='coral', rasterized=True)
axes[1].axhline(0, color='black', linewidth=1, linestyle='--')
axes[1].set_xlabel('Предсказанная цена ($)')
axes[1].set_ylabel('Остаток ($)')
axes[1].set_title('Остатки vs Предсказания', fontsize=13)
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('../reports/figures/residuals_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Средний остаток:  ${residuals.mean():+,.2f}")
print(f"Медиана остатков: ${np.median(residuals):+,.2f}")
print(f"Std остатков:     ${residuals.std():,.2f}")

## 3. Где модель ошибается сильнее

### 3.1 Топ-20 худших предсказаний

In [ ]:
# Собираем результаты вместе с сырыми данными
results_df = test_raw.copy()
results_df = results_df.loc[results_df.index.isin(X_test.index)].copy()

# Приводим y_pred к тому же индексу
pred_series = pd.Series(y_pred, index=X_test.index)
results_df['price_pred'] = pred_series
results_df['error_abs'] = np.abs(results_df['price'] - results_df['price_pred'])
results_df['error_pct'] = (results_df['error_abs'] / results_df['price'].clip(lower=1)) * 100
results_df['error_signed'] = results_df['price'] - results_df['price_pred']  # > 0 — недооценка

print(f"Записей для анализа: {len(results_df):,}")

worst_20 = results_df.nlargest(20, 'error_abs')[
    ['year', 'manufacturer', 'model', 'condition', 'odometer',
     'price', 'price_pred', 'error_abs', 'error_pct']
]
worst_20['price']      = worst_20['price'].round(0).astype(int)
worst_20['price_pred'] = worst_20['price_pred'].round(0).astype(int)
worst_20['error_abs']  = worst_20['error_abs'].round(0).astype(int)
worst_20['error_pct']  = worst_20['error_pct'].round(1)

print(f"\nТоп-20 худших предсказаний (по абсолютной ошибке):")
print(worst_20.to_string(index=False))

In [ ]:
# Анализ паттернов в худших предсказаниях
worst_100 = results_df.nlargest(100, 'error_abs')

print("Паттерны в топ-100 худших предсказаниях:")
print(f"  Средний год: {worst_100['year'].mean():.0f} (vs {results_df['year'].mean():.0f} в целом)")
print(f"  Средний пробег: {worst_100['odometer'].mean():,.0f} (vs {results_df['odometer'].mean():,.0f} в целом)")
print(f"  Средняя цена: ${worst_100['price'].mean():,.0f} (vs ${results_df['price'].mean():,.0f} в целом)")
print(f"  Доля недооценки: {(worst_100['error_signed'] > 0).mean():.0%}")
print(f"  Доля переоценки: {(worst_100['error_signed'] < 0).mean():.0%}")

## 4. Ошибки в разрезе категорий

### 4.1 По производителю

In [ ]:
manuf_stats = results_df.groupby('manufacturer').agg(
    count   = ('price', 'count'),
    MAE     = ('error_abs', 'mean'),
    MAPE    = ('error_pct', 'mean')
).reset_index()

# Топ-15 производителей по количеству объявлений
top_manufacturers = manuf_stats.nlargest(15, 'count')['manufacturer'].tolist()
manuf_top = manuf_stats[manuf_stats['manufacturer'].isin(top_manufacturers)].sort_values('MAE', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(data=manuf_top, x='MAE',  y='manufacturer', ax=axes[0], palette='viridis_r')
axes[0].set_title('MAE по производителям (топ-15 по количеству)', fontsize=12)
axes[0].set_xlabel('MAE ($)')
axes[0].set_ylabel('')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

sns.barplot(data=manuf_top, x='MAPE', y='manufacturer', ax=axes[1], palette='flare')
axes[1].set_title('MAPE по производителям (топ-15 по количеству)', fontsize=12)
axes[1].set_xlabel('MAPE (%)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../reports/figures/error_by_manufacturer.png', dpi=150, bbox_inches='tight')
plt.show()

print("Топ-5 производителей с наибольшей MAE:")
print(manuf_top.head(5)[['manufacturer', 'count', 'MAE', 'MAPE']].to_string(index=False))

### 4.2 По возрасту автомобиля

In [ ]:
CURRENT_YEAR = 2022
results_df['car_age'] = CURRENT_YEAR - results_df['year'].fillna(results_df['year'].median())

results_df['age_group'] = pd.cut(
    results_df['car_age'],
    bins=[0, 5, 10, 15, 20, 100],
    labels=['0-5 лет', '6-10 лет', '11-15 лет', '16-20 лет', '20+ лет']
)

age_stats = results_df.groupby('age_group', observed=False).agg(
    count = ('price', 'count'),
    MAE   = ('error_abs', 'mean'),
    MAPE  = ('error_pct', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=age_stats, x='age_group', y='MAE',  ax=axes[0], palette='viridis_r')
axes[0].set_title('MAE по возрасту автомобиля', fontsize=13)
axes[0].set_xlabel('Возрастная группа')
axes[0].set_ylabel('MAE ($)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

sns.barplot(data=age_stats, x='age_group', y='MAPE', ax=axes[1], palette='flare')
axes[1].set_title('MAPE по возрасту автомобиля', fontsize=13)
axes[1].set_xlabel('Возрастная группа')
axes[1].set_ylabel('MAPE (%)')

plt.tight_layout()
plt.savefig('../reports/figures/error_by_age.png', dpi=150, bbox_inches='tight')
plt.show()

print(age_stats.to_string(index=False))

### 4.3 По состоянию автомобиля

In [ ]:
results_df['condition_clean'] = results_df['condition'].fillna('unknown')

cond_stats = results_df.groupby('condition_clean').agg(
    count = ('price', 'count'),
    MAE   = ('error_abs', 'mean'),
    MAPE  = ('error_pct', 'mean'),
    median_price = ('price', 'median')
).reset_index()
cond_stats = cond_stats[cond_stats['count'] >= 50].sort_values('MAE', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=cond_stats, x='condition_clean', y='MAE', palette='viridis_r', ax=ax)
ax.set_title('MAE по состоянию автомобиля', fontsize=13)
ax.set_xlabel('Состояние')
ax.set_ylabel('MAE ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('../reports/figures/error_by_condition.png', dpi=150)
plt.show()

print(cond_stats[['condition_clean', 'count', 'MAE', 'MAPE', 'median_price']].to_string(index=False))

## 5. Важность признаков (Permutation Importance)

**Permutation Importance** — надёжный метод оценки вклада каждого признака.
Признак "перемешивается" случайно; если качество модели сильно падает — признак важен.

Работает для любой модели (в отличие от `feature_importances_` — только для деревьев).

In [ ]:
# Используем выборку для ускорения вычислений
SAMPLE_SIZE = min(3000, len(X_test))
idx_sample  = np.random.default_rng(42).choice(len(X_test), size=SAMPLE_SIZE, replace=False)
X_sample    = X_test.iloc[idx_sample]
y_sample    = y_test_log.iloc[idx_sample]

print(f"Вычисление Permutation Importance на {SAMPLE_SIZE} объектах...")

perm = permutation_importance(
    model, X_sample, y_sample,
    n_repeats=10,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    'Feature':    X_test.columns,
    'Importance': perm.importances_mean,
    'Std':        perm.importances_std
}).sort_values('Importance', ascending=False)

print("\nТоп-15 важных признаков:")
print(importance_df.head(15).to_string(index=False))

In [ ]:
top_imp = importance_df.head(15).sort_values('Importance')

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_imp['Feature'], top_imp['Importance'],
        xerr=top_imp['Std'], capsize=4,
        color='steelblue', alpha=0.85, edgecolor='white')
ax.set_xlabel('Увеличение MAE при перестановке (важность)', fontsize=12)
ax.set_title('Топ-15 важных признаков (Permutation Importance)', fontsize=14)

plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Признаки с нулевой или отрицательной важностью
zero_imp = importance_df[importance_df['Importance'] <= 0]
print(f"Признаков с нулевой/отрицательной важностью: {len(zero_imp)} из {len(importance_df)}")
if len(zero_imp) > 0:
    print("Это кандидаты на удаление в следующей итерации:")
    print(zero_imp['Feature'].tolist())

## 6. Итоговые выводы

In [ ]:
top3_features = importance_df.head(3)['Feature'].tolist()
top3_imp      = importance_df.head(3)['Importance'].tolist()

worst_age_group = age_stats.sort_values('MAE', ascending=False).iloc[0]['age_group']
best_age_group  = age_stats.sort_values('MAE').iloc[0]['age_group']

print("=" * 65)
print("ИТОГОВЫЙ АНАЛИЗ ОШИБОК")
print("=" * 65)

print(f"""
Метрики финальной модели на тестовой выборке:
  MAE:  ${mae:>10,.2f}
  RMSE: ${rmse:>10,.2f}
  R²:    {r2:>10.4f}
  MAPE:  {mape:>10.2f}%

Топ-3 важных признака:
  1. {top3_features[0]} (importance={top3_imp[0]:.4f})
  2. {top3_features[1]} (importance={top3_imp[1]:.4f})
  3. {top3_features[2]} (importance={top3_imp[2]:.4f})

Где модель ошибается больше всего:
  - Возрастная группа с наибольшей ошибкой: {worst_age_group}
  - Возрастная группа с наименьшей ошибкой: {best_age_group}
  - Автомобили с экстремальным пробегом (очень низким или очень высоким)
  - Раритетные и коллекционные автомобили

Ограничения модели:
  - Не учитывает историю обслуживания и аварийность
  - Не различает комплектации внутри одной модели
  - Коллекционная ценность раритетов не отражена в признаках
  - Цена объявления ≠ цена реальной продажи

Направления улучшения:
  - Добавить TF-IDF признаки из описания объявления
  - Применить признак photo_count (количество фото)
  - Использовать более детальную обработку поля model
  - Попробовать LGBM или нейросетевые подходы
""")
print("=" * 65)